# ZynNova：极复杂多孔多相全电池微结构与 COMSOL MPHTXT 验证

本 notebook 对 `zynnova.zynmorph` 中新增的 COMSOL 导出接口进行完整压力测试。MPHTXT 并不是重新猜测格式，而是直接复用原 `zynsim` 的两套原生写入器：

- 已物化 Tet4 网格：`write_comsol_mphtxt`，保留显式三角边界、材料界面、几何实体和 Selection 对象；
- 超大体素网格：`write_large_voxel_comsol_mphtxt`，以有界额外内存流式写出 Hex8 或六 Tet4/体素。

默认样例包含：

- **110 个独立随机非凸活性颗粒**（55 个负极、55 个正极）；
- 正负极多孔电解液、隔膜电解液；
- 正负极 CBD 网络；
- SEI 与 CEI 界面层；
- 颗粒裂纹与内部微孔；
- 正负极集流体；
- 共 **12 个物理相**，并带大量内部材料界面；
- 完整体素结构流式 Hex8 MPHTXT；
- 横跨全电池厚度的复杂 ROI Tet4 MPHTXT、VTK、Gmsh MSH、Abaqus INP。

数组约定必须注意：旧体素生成器使用 `(x, y, z)`，ZynMorph 使用 `(z, y, x)`；新增适配器会在流式 COMSOL 导出时显式转换，保证 COMSOL 中仍是物理 `x, y, z`。

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from scipy.ndimage import binary_dilation, gaussian_filter, label as connected_label

# 允许 notebook 放在仓库根目录或 notebooks/ 下运行；已 pip install -e 时不会改 sys.path。
try:
    import zynnova
except ModuleNotFoundError:
    candidates = [Path.cwd(), *Path.cwd().parents]
    project_root = next(
        (candidate for candidate in candidates if (candidate / "src" / "zynnova").is_dir()),
        None,
    )
    if project_root is None:
        raise RuntimeError("没有找到 ZynNova 仓库根目录；请先执行 pip install -e .")
    sys.path.insert(0, str(project_root / "src"))
    import zynnova

from zynnova.zynmorph import (
    BatteryPhase,
    MicrostructureVolume,
    analyze_microstructure,
    export_fem_mesh,
    export_voxel_comsol_mphtxt,
    inspect_comsol_mphtxt,
    mesh_microstructure,
    plan_voxel_comsol_mphtxt,
)
from zynnova.zynsim.battery.microstructure import (
    ExplicitParticleFullCellConfig,
    NEGATIVE_ACTIVE as SOURCE_NEGATIVE_ACTIVE,
    NEGATIVE_ELECTROLYTE as SOURCE_NEGATIVE_ELECTROLYTE,
    POSITIVE_ACTIVE as SOURCE_POSITIVE_ACTIVE,
    POSITIVE_ELECTROLYTE as SOURCE_POSITIVE_ELECTROLYTE,
    SEPARATOR_ELECTROLYTE as SOURCE_SEPARATOR_ELECTROLYTE,
    generate_explicit_particle_full_cell,
)

print("ZynNova version:", zynnova.__version__)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)

## 1. 高复杂度配置

默认配置本身就是高复杂度测试，而不是玩具立方体。`WRITE_FULL_STREAMED_TET4` 默认关闭，是因为完整六 Tet4/体素文本会显著大于 Hex8；打开后仍使用原来的有界内存流式写入器。

In [ ]:
SEED = 20260817
VOXEL_SIZE_XYZ_M = (0.45e-6, 0.45e-6, 0.45e-6)
BASE_SHAPE_XYZ = (56, 56, 56)
LAYER_VOXELS_X = (24, 8, 24)
COLLECTOR_THICKNESS_VOXELS = 2
NEGATIVE_PARTICLE_COUNT = 55
POSITIVE_PARTICLE_COUNT = 55

WRITE_FULL_HEX8 = True
WRITE_ROI_TET4 = True
WRITE_FULL_STREAMED_TET4 = False

OUTPUT_ROOT = Path("zynnova_runs/zynmorph_complex_mphtxt")
RUN_DIRECTORY = OUTPUT_ROOT / f"complex_full_cell_seed_{SEED}"
if RUN_DIRECTORY.exists():
    shutil.rmtree(RUN_DIRECTORY)
RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)

print("Output directory:", RUN_DIRECTORY.resolve())
print("Base voxels:", int(np.prod(BASE_SHAPE_XYZ)))
print("Requested particles:", NEGATIVE_PARTICLE_COUNT + POSITIVE_PARTICLE_COUNT)

## 2. 复杂形貌辅助函数

以下升级不是随机撒点：先用旧 `zynsim` 中成熟的非凸颗粒生成器产生颗粒分辨全电池，再在其物理相上构造相关 CBD、界面层、裂纹和内部椭球孔。所有操作均保持整数材料标签。

In [ ]:
def smooth_random_field(
    shape: tuple[int, int, int],
    sigma: tuple[float, float, float],
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return gaussian_filter(rng.normal(size=shape), sigma=sigma, mode="reflect")


def select_ranked_fraction(
    mask: np.ndarray,
    score: np.ndarray,
    fraction: float,
    *,
    minimum: int = 1,
) -> np.ndarray:
    """在候选 mask 内按 score 精确选取给定比例，避免阈值导致相完全消失。"""
    indices = np.flatnonzero(mask)
    selected = np.zeros(mask.size, dtype=bool)
    if indices.size == 0:
        return selected.reshape(mask.shape)
    count = min(indices.size, max(minimum, int(round(indices.size * fraction))))
    if count == indices.size:
        chosen = indices
    else:
        local_scores = score.ravel()[indices]
        chosen = indices[np.argpartition(local_scores, -count)[-count:]]
    selected[chosen] = True
    return selected.reshape(mask.shape)


def particle_crack_and_void_mask(
    active_mask: np.ndarray,
    *,
    plane_count: int,
    void_count: int,
    seed: int,
) -> np.ndarray:
    """生成穿过多个颗粒的粗糙斜裂纹，并叠加颗粒内部椭球孔。"""
    rng = np.random.default_rng(seed)
    shape = np.asarray(active_mask.shape, dtype=float)
    coordinates = np.indices(active_mask.shape, dtype=float)
    for axis in range(3):
        coordinates[axis] /= max(shape[axis] - 1.0, 1.0)

    modulation = smooth_random_field(active_mask.shape, (1.2, 1.8, 1.8), seed + 97)
    threshold = float(np.quantile(modulation[active_mask], 0.46))
    cracks = np.zeros(active_mask.shape, dtype=bool)
    for _ in range(plane_count):
        normal = rng.normal(size=3)
        normal /= np.linalg.norm(normal)
        center = rng.uniform(0.25, 0.75, size=3)
        half_width = float(rng.uniform(0.010, 0.022))
        distance = np.abs(
            sum(normal[axis] * (coordinates[axis] - center[axis]) for axis in range(3))
        )
        cracks |= active_mask & (distance < half_width) & (modulation > threshold)

    candidates = np.argwhere(active_mask)
    if candidates.size:
        chosen_centers = candidates[
            rng.choice(len(candidates), size=min(void_count, len(candidates)), replace=False)
        ]
        for center in chosen_centers:
            radii = rng.uniform((1.0, 1.2, 1.2), (2.2, 3.0, 3.0))
            slices = []
            for axis in range(3):
                lower = max(0, int(center[axis] - radii[axis] - 1))
                upper = min(active_mask.shape[axis], int(center[axis] + radii[axis] + 2))
                slices.append(slice(lower, upper))
            local_shape = tuple(item.stop - item.start for item in slices)
            grid = np.indices(local_shape, dtype=float)
            distance_squared = np.zeros(local_shape, dtype=float)
            for axis, item in enumerate(slices):
                distance_squared += (
                    (grid[axis] + item.start - center[axis]) / radii[axis]
                ) ** 2
            local = tuple(slices)
            cracks[local] |= active_mask[local] & (distance_squared <= 1.0)
    return cracks


def percolates_between_opposite_faces(mask: np.ndarray, axis: int) -> bool:
    components, _ = connected_label(mask)
    low = np.unique(np.take(components, 0, axis=axis))
    high = np.unique(np.take(components, -1, axis=axis))
    return bool(set(low[low > 0].tolist()) & set(high[high > 0].tolist()))

In [ ]:
def build_twelve_phase_full_cell(base) -> tuple[np.ndarray, dict[str, object]]:
    """将五相颗粒分辨结构升级为 ZynMorph 的十二相全电池，返回 (x,y,z)。"""
    phase_map = {
        SOURCE_NEGATIVE_ACTIVE: int(BatteryPhase.NEGATIVE_ACTIVE),
        SOURCE_NEGATIVE_ELECTROLYTE: int(BatteryPhase.NEGATIVE_ELECTROLYTE),
        SOURCE_SEPARATOR_ELECTROLYTE: int(BatteryPhase.SEPARATOR_ELECTROLYTE),
        SOURCE_POSITIVE_ELECTROLYTE: int(BatteryPhase.POSITIVE_ELECTROLYTE),
        SOURCE_POSITIVE_ACTIVE: int(BatteryPhase.POSITIVE_ACTIVE),
    }
    core = np.empty_like(base.phase_labels, dtype=np.int32)
    for source_phase, target_phase in phase_map.items():
        core[base.phase_labels == source_phase] = target_phase

    collector = COLLECTOR_THICKNESS_VOXELS
    labels = np.empty((core.shape[0] + 2 * collector, *core.shape[1:]), dtype=np.int32)
    labels[collector:-collector] = core
    labels[:collector] = int(BatteryPhase.NEGATIVE_CURRENT_COLLECTOR)
    labels[-collector:] = int(BatteryPhase.POSITIVE_CURRENT_COLLECTOR)

    negative_active = labels == int(BatteryPhase.NEGATIVE_ACTIVE)
    positive_active = labels == int(BatteryPhase.POSITIVE_ACTIVE)
    negative_pore = labels == int(BatteryPhase.NEGATIVE_ELECTROLYTE)
    positive_pore = labels == int(BatteryPhase.POSITIVE_ELECTROLYTE)

    # 一体素界面邻域中选择粗糙、非均匀 SEI/CEI，而不是把所有界面都涂成同厚度。
    negative_shell = binary_dilation(negative_active, iterations=1) & negative_pore
    positive_shell = binary_dilation(positive_active, iterations=1) & positive_pore
    sei = select_ranked_fraction(
        negative_shell,
        smooth_random_field(labels.shape, (1.5, 2.0, 2.0), SEED + 101),
        0.52,
    )
    cei = select_ranked_fraction(
        positive_shell,
        smooth_random_field(labels.shape, (1.5, 2.0, 2.0), SEED + 102),
        0.48,
    )
    labels[sei] = int(BatteryPhase.NEGATIVE_SEI)
    labels[cei] = int(BatteryPhase.POSITIVE_CEI)

    # CBD 由长相关随机场和靠近活性颗粒的偏置共同决定，得到团簇/细颈混合网络。
    negative_pore = labels == int(BatteryPhase.NEGATIVE_ELECTROLYTE)
    positive_pore = labels == int(BatteryPhase.POSITIVE_ELECTROLYTE)
    negative_near_active = binary_dilation(negative_active, iterations=2) & negative_pore
    positive_near_active = binary_dilation(positive_active, iterations=2) & positive_pore
    negative_cbd_score = smooth_random_field(
        labels.shape, (4.0, 2.2, 2.2), SEED + 103
    ) + 0.8 * negative_near_active
    positive_cbd_score = smooth_random_field(
        labels.shape, (4.0, 2.2, 2.2), SEED + 104
    ) + 0.8 * positive_near_active
    negative_cbd = select_ranked_fraction(negative_pore, negative_cbd_score, 0.13)
    positive_cbd = select_ranked_fraction(positive_pore, positive_cbd_score, 0.15)
    labels[negative_cbd] = int(BatteryPhase.NEGATIVE_CBD)
    labels[positive_cbd] = int(BatteryPhase.POSITIVE_CBD)

    # 斜裂纹 + 颗粒内椭球孔统一放入 crack/void 相；不覆盖电解液、CBD 或界面层。
    negative_active = labels == int(BatteryPhase.NEGATIVE_ACTIVE)
    positive_active = labels == int(BatteryPhase.POSITIVE_ACTIVE)
    damage = particle_crack_and_void_mask(
        negative_active, plane_count=4, void_count=14, seed=SEED + 105
    ) | particle_crack_and_void_mask(
        positive_active, plane_count=6, void_count=18, seed=SEED + 106
    )
    labels[damage] = int(BatteryPhase.CRACK)

    unique, counts = np.unique(labels, return_counts=True)
    diagnostics = {
        "phase_counts_xyz": {int(key): int(value) for key, value in zip(unique, counts, strict=True)},
        "sei_voxels": int(np.count_nonzero(sei)),
        "cei_voxels": int(np.count_nonzero(cei)),
        "negative_cbd_voxels": int(np.count_nonzero(negative_cbd)),
        "positive_cbd_voxels": int(np.count_nonzero(positive_cbd)),
        "damage_voxels": int(np.count_nonzero(damage)),
    }
    return labels, diagnostics

## 3. 生成 110 个非凸颗粒的五相基础结构

该生成器来自此前保留的 `zynsim.battery.microstructure`，包含多叶片非凸形状、表面粗糙度、正极内部孔与裂纹概率、目标活性体积分数校准及电解液贯通检查。

In [ ]:
base_config = ExplicitParticleFullCellConfig(
    voxel_shape=BASE_SHAPE_XYZ,
    voxel_size_m=VOXEL_SIZE_XYZ_M,
    layer_voxels=LAYER_VOXELS_X,
    negative_particle_count=NEGATIVE_PARTICLE_COUNT,
    positive_particle_count=POSITIVE_PARTICLE_COUNT,
    negative_active_fraction=0.46,
    positive_active_fraction=0.45,
    negative_lobes=(2, 5),
    positive_lobes=(5, 10),
    negative_roughness=0.16,
    positive_roughness=0.24,
    positive_internal_pore_probability=0.85,
    positive_crack_probability=0.75,
    calibration_iterations=4,
    minimum_particle_voxels=5,
    seed=SEED,
)

started = time.perf_counter()
base = generate_explicit_particle_full_cell(base_config)
base_generation_seconds = time.perf_counter() - started

assert base.particle_counts == {
    "negative": NEGATIVE_PARTICLE_COUNT,
    "positive": POSITIVE_PARTICLE_COUNT,
}
assert base.electrolyte_percolation["full_cell_electrolyte_x"]

base_summary = pd.DataFrame(
    [
        {"quantity": "generation_seconds", "value": base_generation_seconds},
        {"quantity": "negative_particles", "value": base.particle_counts["negative"]},
        {"quantity": "positive_particles", "value": base.particle_counts["positive"]},
        {"quantity": "total_particles", "value": len(base.particles)},
        {"quantity": "negative_active_fraction", "value": base.achieved_active_fractions["negative"]},
        {"quantity": "positive_active_fraction", "value": base.achieved_active_fractions["positive"]},
        {"quantity": "full_cell_electrolyte_x", "value": base.electrolyte_percolation["full_cell_electrolyte_x"]},
    ]
)
display(base_summary)

## 4. 升级为 12 相复杂多孔全电池

添加集流体、相关 CBD、非均匀 SEI/CEI、贯穿颗粒的斜裂纹和内部椭球孔。随后从旧生成器的 `(x,y,z)` 显式转为 ZynMorph 的 `(z,y,x)`。

In [ ]:
started = time.perf_counter()
labels_xyz, upgrade_diagnostics = build_twelve_phase_full_cell(base)
upgrade_seconds = time.perf_counter() - started

expected_phases = {int(phase) for phase in BatteryPhase}
observed_phases = set(map(int, np.unique(labels_xyz)))
assert observed_phases == expected_phases, (observed_phases, expected_phases)

# ZynMorph 约定：(z, y, x)；同时将旧 spacing 的 (x,y,z) 反转为 (z,y,x)。
spacing_xyz = tuple(float(value) for value in VOXEL_SIZE_XYZ_M)
spacing_zyx = (spacing_xyz[2], spacing_xyz[1], spacing_xyz[0])
labels_zyx = np.ascontiguousarray(labels_xyz.transpose(2, 1, 0))
volume = MicrostructureVolume(
    labels=labels_zyx,
    voxel_size_m=spacing_zyx,
    metadata={
        "generator": "explicit-particle-plus-multiphase-upgrade",
        "seed": SEED,
        "particle_count": len(base.particles),
        "source_shape_xyz": BASE_SHAPE_XYZ,
        "collector_thickness_voxels": COLLECTOR_THICKNESS_VOXELS,
        **upgrade_diagnostics,
    },
)

volume_path = volume.save_npz(RUN_DIRECTORY / "complex_full_cell_12_phase.npz")
print("Upgrade seconds:", round(upgrade_seconds, 4))
print("ZynMorph shape (z,y,x):", volume.shape)
print("Physical size (z,y,x), um:", np.asarray(volume.physical_size_m) * 1e6)
print("Saved volume:", volume_path.resolve())

In [ ]:
phase_rows = []
unique, counts = np.unique(volume.labels, return_counts=True)
for phase_id, count in zip(unique, counts, strict=True):
    phase_rows.append(
        {
            "phase_id": int(phase_id),
            "phase_name": volume.phase_names[int(phase_id)],
            "voxels": int(count),
            "volume_fraction": float(count / volume.labels.size),
            "volume_um3": float(count * np.prod(volume.voxel_size_m) * 1e18),
        }
    )
phase_table = pd.DataFrame(phase_rows).sort_values("phase_id").reset_index(drop=True)
display(phase_table)
assert len(phase_table) == 12
assert np.isclose(phase_table["volume_fraction"].sum(), 1.0)

## 5. 正交截面与三维相分布

颜色只用于区分离散材料相，不参与任何网格或 COMSOL 实体编号。

In [ ]:
phase_colors = [
    "#7fcdbb",  # separator electrolyte
    "#d73027",  # positive active
    "#74add1",  # positive electrolyte
    "#4d4d4d",  # negative active
    "#abd9e9",  # negative electrolyte
    "#fdae61",  # positive CBD
    "#a6611a",  # negative CBD
    "#f46d43",  # CEI
    "#8073ac",  # SEI
    "#000000",  # crack/void
    "#bdbdbd",  # negative collector
    "#fdd835",  # positive collector
]
cmap = ListedColormap(phase_colors)
norm = BoundaryNorm(np.arange(-0.5, 12.5, 1.0), cmap.N)

z_mid, y_mid, x_mid = (size // 2 for size in volume.shape)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
axes[0].imshow(volume.labels[z_mid, :, :], cmap=cmap, norm=norm, origin="lower", aspect="auto")
axes[0].set_title(f"z = {z_mid}: y-x full-cell section")
axes[0].set_xlabel("x voxel (through thickness)")
axes[0].set_ylabel("y voxel")
axes[1].imshow(volume.labels[:, y_mid, :], cmap=cmap, norm=norm, origin="lower", aspect="auto")
axes[1].set_title(f"y = {y_mid}: z-x full-cell section")
axes[1].set_xlabel("x voxel (through thickness)")
axes[1].set_ylabel("z voxel")
axes[2].imshow(volume.labels[:, :, x_mid], cmap=cmap, norm=norm, origin="lower")
axes[2].set_title(f"x = {x_mid}: z-y in-plane section")
axes[2].set_xlabel("y voxel")
axes[2].set_ylabel("z voxel")
plt.show()

In [ ]:
# 为避免遮挡，每个非电解液相最多绘制 1,250 个体素中心。
rng = np.random.default_rng(SEED + 200)
fig = plt.figure(figsize=(11, 9))
axis = fig.add_subplot(111, projection="3d")
phases_for_3d = [1, 3, 5, 6, 7, 8, 9, 10, 11]
for phase_id in phases_for_3d:
    coordinates_zyx = np.argwhere(volume.labels == phase_id)
    if len(coordinates_zyx) > 1250:
        coordinates_zyx = coordinates_zyx[
            rng.choice(len(coordinates_zyx), size=1250, replace=False)
        ]
    axis.scatter(
        coordinates_zyx[:, 2],
        coordinates_zyx[:, 1],
        coordinates_zyx[:, 0],
        s=3,
        alpha=0.55,
        c=phase_colors[phase_id],
        label=volume.phase_names[phase_id],
    )
axis.set_xlabel("x voxel")
axis.set_ylabel("y voxel")
axis.set_zlabel("z voxel")
axis.set_title("Downsampled solid/interphase/damage architecture")
axis.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8)
plt.show()

## 6. 形貌、连通性与孔隙贯通审计

单个电解液相在不同层使用不同标签，因此还单独检查其并集在去除集流体后是否沿电池厚度方向贯通。

In [ ]:
started = time.perf_counter()
metrics = analyze_microstructure(volume)
metrics_seconds = time.perf_counter() - started

metric_rows = []
for phase_id, item in sorted(metrics.phases.items()):
    metric_rows.append(
        {
            "phase_id": phase_id,
            "phase_name": volume.phase_names[phase_id],
            "volume_fraction": item.volume_fraction,
            "largest_component_fraction": item.connected_fraction,
            "percolates_z": item.percolates[0],
            "percolates_y": item.percolates[1],
            "percolates_x": item.percolates[2],
            "specific_surface_area_per_m": item.specific_surface_area_per_m,
        }
    )
metric_table = pd.DataFrame(metric_rows)
display(metric_table)

collector = COLLECTOR_THICKNESS_VOXELS
core_xyz = labels_xyz[collector:-collector]
electrolyte_union_xyz = np.isin(
    core_xyz,
    [
        int(BatteryPhase.SEPARATOR_ELECTROLYTE),
        int(BatteryPhase.POSITIVE_ELECTROLYTE),
        int(BatteryPhase.NEGATIVE_ELECTROLYTE),
    ],
)
pore_and_damage_union_xyz = electrolyte_union_xyz | (
    core_xyz == int(BatteryPhase.CRACK)
)
assert percolates_between_opposite_faces(electrolyte_union_xyz, axis=0)
assert percolates_between_opposite_faces(pore_and_damage_union_xyz, axis=0)

print("Metrics seconds:", round(metrics_seconds, 4))
print("Electrolyte union percolates through core x:", True)
print("Electrolyte + damage pore union percolates through core x:", True)
print("Number of distinct material interfaces:", len(metrics.interface_area_m2))

In [ ]:
interface_table = pd.DataFrame(
    [
        {
            "phase_a": pair[0],
            "name_a": volume.phase_names[pair[0]],
            "phase_b": pair[1],
            "name_b": volume.phase_names[pair[1]],
            "area_um2": area * 1e12,
        }
        for pair, area in metrics.interface_area_m2.items()
    ]
).sort_values("area_um2", ascending=False).reset_index(drop=True)
display(interface_table.head(30))

## 7. 先规划，再流式写出完整 Hex8 COMSOL MPHTXT

规划器与写入器都来自旧 `zynsim` 的真实实现。流式路径不会先构建完整节点数组和单元连接数组；额外峰值内存由 `chunk_size` 控制。

In [ ]:
hex_plan = plan_voxel_comsol_mphtxt(
    volume,
    element_type="hex8",
    include_exterior_boundaries=True,
    include_material_interfaces=True,
)
display(pd.DataFrame([asdict(hex_plan)]).T.rename(columns={0: "value"}))

assert hex_plan.vertex_count == np.prod(np.asarray(volume.shape)[::-1] + 1)
assert hex_plan.volume_element_count == volume.labels.size
assert hex_plan.interface_face_count > 0

In [ ]:
full_hex_path = RUN_DIRECTORY / "complex_full_cell_hex8.mphtxt"
full_hex_report = None
full_hex_info = None
if WRITE_FULL_HEX8:
    started = time.perf_counter()
    full_hex_report = export_voxel_comsol_mphtxt(
        full_hex_path,
        volume,
        element_type="hex8",
        include_exterior_boundaries=True,
        include_material_interfaces=True,
        chunk_size=65_536,
        prefer_native=True,
        verify=True,
    )
    full_hex_seconds = time.perf_counter() - started
    full_hex_info = inspect_comsol_mphtxt(full_hex_path)

    assert full_hex_info.format_version == (0, 1)
    assert full_hex_info.mesh_class_version == 4
    assert full_hex_info.space_dimension == 3
    assert full_hex_info.start_vertex_index == 0
    assert full_hex_info.element_counts["hex"] == volume.labels.size
    assert full_hex_info.element_counts.get("quad", 0) > 0

    selection_table = pd.DataFrame(
        [
            {
                "label": item.label,
                "dimension": item.dimension,
                "entity_count": len(item.entity_ids),
                "entities": item.entity_ids,
            }
            for item in full_hex_info.selections
        ]
    )
    print("Full Hex8 MPHTXT:", full_hex_path.resolve())
    print("Export seconds:", round(full_hex_seconds, 4))
    print("File size MiB:", round(full_hex_path.stat().st_size / 2**20, 3))
    display(selection_table.head(35))

In [ ]:
# 直接检查文件头，验证是原写入器的 COMSOL 0 1 / Mesh-v4 布局。
if WRITE_FULL_HEX8:
    file_lines = full_hex_path.read_text(encoding="utf-8").splitlines()
    header_preview = file_lines[:24]
    print("\n".join(header_preview))
    assert file_lines[0] == "# COMSOL native text mesh generated by ZynNova ZynSim"
    assert "0 1" in file_lines[:8]
    # Selection 很多时 Mesh 对象会排在完整 tag/type 表之后，不能只看前 24 行。
    assert any("Mesh" in line for line in file_lines[:256])

## 8. 构造横跨完整厚度的复杂 Tet4 ROI

ROI 保留负集流体、负极、隔膜、正极和正集流体，只在两个横向方向裁剪，因此仍包含全部 12 相。每个体素按一致体对角线拆为六个共形 Tet4；显式提取外边界与所有材料界面，写入原生 COMSOL Selection 对象。

In [ ]:
roi_z = slice(18, 38)
roi_y = slice(18, 38)
roi_x = slice(None)
roi_labels = np.ascontiguousarray(volume.labels[roi_z, roi_y, roi_x])
dz, dy, dx = volume.voxel_size_m
oz, oy, ox = volume.origin_m
roi_volume = MicrostructureVolume(
    labels=roi_labels,
    voxel_size_m=volume.voxel_size_m,
    origin_m=(oz + roi_z.start * dz, oy + roi_y.start * dy, ox),
    phase_names=volume.phase_names,
    metadata={"source": "full-thickness-central-ROI", "parent": str(volume_path)},
)
assert set(map(int, np.unique(roi_volume.labels))) == expected_phases

started = time.perf_counter()
roi_fem = mesh_microstructure(
    roi_volume,
    maximum_tetrahedra=roi_volume.labels.size * 6,
)
roi_meshing_seconds = time.perf_counter() - started

assert roi_fem.quality.fem_ready
assert roi_fem.quality.inverted_cells == 0
assert roi_fem.quality.degenerate_cells == 0
assert roi_fem.mesh.n_cells == roi_volume.labels.size * 6

quality_summary = pd.DataFrame(
    [
        {
            "roi_shape_zyx": roi_volume.shape,
            "nodes": roi_fem.mesh.n_nodes,
            "tetrahedra": roi_fem.mesh.n_cells,
            "inverted": roi_fem.quality.inverted_cells,
            "degenerate": roi_fem.quality.degenerate_cells,
            "minimum_mean_ratio": roi_fem.quality.minimum_mean_ratio,
            "median_mean_ratio": roi_fem.quality.median_mean_ratio,
            "meshing_seconds": roi_meshing_seconds,
        }
    ]
)
display(quality_summary)

In [ ]:
roi_export_directory = RUN_DIRECTORY / "full_thickness_roi_mesh"
roi_domain_unions = {
    "solid_skeleton": (
        int(BatteryPhase.POSITIVE_ACTIVE),
        int(BatteryPhase.NEGATIVE_ACTIVE),
        int(BatteryPhase.POSITIVE_CBD),
        int(BatteryPhase.NEGATIVE_CBD),
        int(BatteryPhase.POSITIVE_CEI),
        int(BatteryPhase.NEGATIVE_SEI),
        int(BatteryPhase.NEGATIVE_CURRENT_COLLECTOR),
        int(BatteryPhase.POSITIVE_CURRENT_COLLECTOR),
    ),
    "electrolyte_transport_domains": (
        int(BatteryPhase.SEPARATOR_ELECTROLYTE),
        int(BatteryPhase.POSITIVE_ELECTROLYTE),
        int(BatteryPhase.NEGATIVE_ELECTROLYTE),
    ),
    "damage_domain": (int(BatteryPhase.CRACK),),
}

if WRITE_ROI_TET4:
    started = time.perf_counter()
    roi_fem = export_fem_mesh(
        roi_fem,
        roi_export_directory,
        formats=("vtk", "msh", "inp", "mphtxt"),
        export_boundary=True,
        comsol_domain_selections=roi_domain_unions,
        comsol_options={
            "include_boundaries": True,
            "include_internal_interfaces": True,
            "include_exterior": True,
            "create_interface_selections": True,
            "verify": True,
        },
    )
    roi_export_seconds = time.perf_counter() - started
    roi_mphtxt_path = roi_fem.exports["mphtxt"]
    roi_mphtxt_info = inspect_comsol_mphtxt(roi_mphtxt_path)

    assert roi_mphtxt_info.element_counts["tet"] == roi_fem.mesh.n_cells
    assert roi_mphtxt_info.element_counts["tri"] > 0
    roi_selection_labels = {item.label for item in roi_mphtxt_info.selections}
    for required in (
        "solid_skeleton",
        "electrolyte_transport_domains",
        "damage_domain",
        "all_domains",
        "xmin",
        "xmax",
        "all_exterior",
        "x_terminal_pair",
    ):
        assert required in roi_selection_labels, required
    assert any(label.startswith("interface_") for label in roi_selection_labels)

    export_rows = [
        {
            "role": role,
            "path": str(path.resolve()),
            "size_MiB": path.stat().st_size / 2**20,
        }
        for role, path in sorted(roi_fem.exports.items())
    ]
    print("ROI export seconds:", round(roi_export_seconds, 4))
    display(pd.DataFrame(export_rows))

## 9. 可选：完整结构流式六 Tet4/体素 MPHTXT

打开开关后，不会构建 `6 × n_voxel` 的 Python 连接数组；仍由旧 `zynsim` 的流式写入器逐块生成。文件会明显大于 Hex8。

In [ ]:
full_streamed_tet4_path = RUN_DIRECTORY / "complex_full_cell_streamed_tet4.mphtxt"
full_streamed_tet4_report = None
if WRITE_FULL_STREAMED_TET4:
    tet_plan = plan_voxel_comsol_mphtxt(volume, element_type="tet4")
    display(pd.DataFrame([asdict(tet_plan)]).T.rename(columns={0: "value"}))
    started = time.perf_counter()
    full_streamed_tet4_report = export_voxel_comsol_mphtxt(
        full_streamed_tet4_path,
        volume,
        element_type="tet4",
        chunk_size=65_536,
        prefer_native=True,
        verify=True,
    )
    full_streamed_tet4_seconds = time.perf_counter() - started
    full_streamed_tet4_info = inspect_comsol_mphtxt(full_streamed_tet4_path)
    assert full_streamed_tet4_info.element_counts["tet"] == volume.labels.size * 6
    print("Full streamed Tet4 seconds:", round(full_streamed_tet4_seconds, 4))
    print("Full streamed Tet4 MiB:", round(full_streamed_tet4_path.stat().st_size / 2**20, 3))
else:
    print("WRITE_FULL_STREAMED_TET4=False：已跳过超大完整 Tet4 文本；ROI Tet4 已完整验证。")

## 10. 最终硬门禁与 JSON 审计报告

In [ ]:
assert len(base.particles) >= 100
assert observed_phases == expected_phases
assert upgrade_diagnostics["damage_voxels"] > 0
assert upgrade_diagnostics["negative_cbd_voxels"] > 0
assert upgrade_diagnostics["positive_cbd_voxels"] > 0
assert upgrade_diagnostics["sei_voxels"] > 0
assert upgrade_diagnostics["cei_voxels"] > 0
assert volume_path.is_file()
assert roi_fem.quality.fem_ready
if WRITE_FULL_HEX8:
    assert full_hex_path.is_file()
if WRITE_ROI_TET4:
    assert roi_fem.exports["mphtxt"].is_file()

final_report = {
    "schema": "zynnova.zynmorph.complex-comsol-test.v1",
    "seed": SEED,
    "particle_counts": base.particle_counts,
    "total_particles": len(base.particles),
    "shape_xyz_with_collectors": list(map(int, labels_xyz.shape)),
    "shape_zyx": list(map(int, volume.shape)),
    "voxel_size_m_zyx": list(map(float, volume.voxel_size_m)),
    "phase_counts": upgrade_diagnostics["phase_counts_xyz"],
    "phase_count": len(observed_phases),
    "distinct_interface_pairs": len(metrics.interface_area_m2),
    "electrolyte_union_percolates_core_x": True,
    "generation_seconds": base_generation_seconds,
    "upgrade_seconds": upgrade_seconds,
    "metrics_seconds": metrics_seconds,
    "full_hex8": None if not WRITE_FULL_HEX8 else {
        "path": str(full_hex_path.resolve()),
        "size_bytes": full_hex_path.stat().st_size,
        "vertices": full_hex_info.vertex_count,
        "hex8": full_hex_info.element_counts["hex"],
        "boundary_and_interface_quads": full_hex_info.element_counts.get("quad", 0),
        "selection_count": len(full_hex_info.selections),
    },
    "roi_tet4": None if not WRITE_ROI_TET4 else {
        "path": str(roi_fem.exports["mphtxt"].resolve()),
        "nodes": roi_fem.mesh.n_nodes,
        "tetrahedra": roi_fem.mesh.n_cells,
        "triangles": roi_mphtxt_info.element_counts["tri"],
        "selection_count": len(roi_mphtxt_info.selections),
        "inverted": roi_fem.quality.inverted_cells,
        "degenerate": roi_fem.quality.degenerate_cells,
    },
}
report_path = RUN_DIRECTORY / "complex_multiphase_comsol_validation.json"
report_path.write_text(json.dumps(final_report, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(final_report, indent=2, ensure_ascii=False))
print("\n全部硬门禁通过。报告：", report_path.resolve())